In [1]:
import mlflow
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss
from data import load_data, get_splits

In [2]:
X, y = load_data()
X_tr, X_te, y_tr, y_te = get_splits()
print(X.shape, "diabetes yes:", y.sum(), "no:", (y == 0).sum())

(768, 8) diabetes yes: 268 no: 500


In [4]:
params = {"n_estimators": 150, "learning_rate": 0.05, "max_depth": 3, "subsample": 0.8, "random_state": 42}

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("pima_diabetes_gb_notebook")

with mlflow.start_run(run_name="notebook"):
    mlflow.log_params(params)
    gb = GradientBoostingClassifier(**params).fit(X_tr, y_tr)
    pred = gb.predict(X_te)
    metrics = {"accuracy": accuracy_score(y_te, pred),
               "f1": f1_score(y_te, pred),
               "roc_auc": roc_auc_score(y_te, gb.predict_proba(X_te)[:, 1])}
    mlflow.log_metrics(metrics)
metrics

2026/10/17 15:39:41 INFO mlflow.tracking.fluent: Experiment with name 'pima_diabetes_gb_notebook' does not exist. Creating a new experiment.


{'accuracy': 0.7467532467532467,
 'f1': 0.6213592233009708,
 'roc_auc': 0.8274074074074075}

In [5]:
# quick calibration check
for method in ["sigmoid", "isotonic"]:
    cal = CalibratedClassifierCV(GradientBoostingClassifier(**params), method=method, cv=5).fit(X_tr, y_tr)
    print(method, round(brier_score_loss(y_te, cal.predict_proba(X_te)[:, 1]), 4))
print("uncalibrated", round(brier_score_loss(y_te, gb.predict_proba(X_te)[:, 1]), 4))

sigmoid 0.1609
isotonic 0.1604
uncalibrated 0.1647
